# SleepAgent API 单步调试

按顺序逐格执行。默认关闭缓存、失败时抛出异常，并且每次只取 3 条记录，便于观察真实请求。

> 注意：下面的 provider 测试会访问公网 API。请勿把 API key 直接写进 notebook。

In [10]:
# 1. 定位项目根目录，使 notebook 从项目目录或 notebooks/ 目录启动都能工作
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "sleep_ai_scientist").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "sleep_ai_scientist").is_dir():
    raise RuntimeError("找不到 SleepAgent 项目根目录")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def load_env_file(path: Path, *, override: bool = False) -> list[str]:
    """Load simple KEY=VALUE pairs from .env into this notebook kernel."""
    if not path.exists():
        return []

    loaded = []
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and (override or key not in os.environ):
            os.environ[key] = value
            loaded.append(key)
    return loaded

loaded_env_keys = load_env_file(ROOT / ".env")
print(f"Loaded {len(loaded_env_keys)} variables from {ROOT / '.env'}")
ROOT

Loaded 9 variables from /home/jwj/code/SleepAgent/.env


PosixPath('/home/jwj/code/SleepAgent')

In [ ]:
# 2. 打开详细异常信息；异常后可在新单元格执行 %debug 进入事后调试
%xmode Verbose

In [2]:
# 3. 导入 API 入口
from sleep_ai_scientist.api.literature_client import build_client
from sleep_ai_scientist.common.config import load_config

PROVIDERS = ("openalex", "pubmed", "europe_pmc", "semantic_scholar")
PROVIDERS

('openalex', 'pubmed', 'europe_pmc', 'semantic_scholar')

In [3]:
# 4. 加载并覆盖调试配置（只修改内存，不修改 YAML）
config = load_config("configs/grounding_config.yaml")
config["api"].update({
    "enabled": True,
    "cache_enabled": False,  # 每次执行都发真实请求
    "fail_open": False,      # 网络/HTTP 错误直接抛出
    "timeout_seconds": 20,
    "max_retries": 1,        # 单步调试时不等待多轮重试
})

QUERY = "insomnia slow wave EEG"
MAX_RESULTS = 3
config["api"]

{'enabled': True,
 'fail_open': False,
 'cache_enabled': False,
 'cache_dir': '.cache/sleepagent_api',
 'timeout_seconds': 20,
 'max_retries': 1,
 'backoff_seconds': 1.5,
 'max_results_per_query': 20,
 'providers': {'pubmed': {'enabled': True,
   'email_env': 'NCBI_EMAIL',
   'tool_env': 'NCBI_TOOL',
   'api_key_env': 'NCBI_API_KEY',
   'base_url': 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils',
   'requests_per_second_without_key': 3,
   'requests_per_second_with_key': 10},
  'europe_pmc': {'enabled': True,
   'base_url': 'https://www.ebi.ac.uk/europepmc/webservices/rest',
   'email_env': 'EUROPE_PMC_EMAIL',
   'requests_per_second': 3},
  'openalex': {'enabled': True,
   'base_url': 'https://api.openalex.org',
   'api_key_env': 'OPENALEX_API_KEY',
   'requests_per_second': 5},
  'semantic_scholar': {'enabled': True,
   'base_url': 'https://api.semanticscholar.org/graph/v1',
   'api_key_env': 'SEMANTIC_SCHOLAR_API_KEY',
   'requests_per_second': 1}},
 'search_queries': ['insomnia slo

In [11]:
# 5. 仅显示凭据是否存在，不输出 secret
credential_status = {
    "NCBI_EMAIL": bool(os.getenv("NCBI_EMAIL")),
    "NCBI_API_KEY": bool(os.getenv("NCBI_API_KEY")),
    "EUROPE_PMC_EMAIL": bool(os.getenv("EUROPE_PMC_EMAIL")),
    "OPENALEX_API_KEY": bool(os.getenv("OPENALEX_API_KEY")),
    "SEMANTIC_SCHOLAR_API_KEY": bool(os.getenv("SEMANTIC_SCHOLAR_API_KEY")),
}
credential_status

{'NCBI_EMAIL': True,
 'NCBI_API_KEY': True,
 'EUROPE_PMC_EMAIL': True,
 'OPENALEX_API_KEY': True,
 'SEMANTIC_SCHOLAR_API_KEY': False}

## Endpoint 连通性检查

`base_url` 只是 client 拼接请求路径的前缀，不是可直接浏览的网页。直接打开 NCBI base URL 通常返回 `404`，Europe PMC base URL 通常返回 `405`；这不表示 API 不可用。

实际搜索 endpoint：PubMed 使用 `.../esearch.fcgi`，Europe PMC 使用 `.../search`。

In [12]:
# 6. 检查实际 endpoint；失败信息可用于区分代理、DNS、TLS 和 HTTP 问题
import requests

connectivity_checks = {
    "pubmed": (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        {"db": "pubmed", "term": QUERY, "retmode": "json", "retmax": 1, "tool": "SleepAgent"},
    ),
    "europe_pmc": (
        "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
        {"query": QUERY, "format": "json", "pageSize": 1},
    ),
}

connectivity = {}
for provider, (url, params) in connectivity_checks.items():
    try:
        response = requests.get(url, params=params, timeout=20)
        connectivity[provider] = {
            "status_code": response.status_code,
            "ok": response.ok,
            "final_url": response.url,
            "response_preview": response.text[:120],
        }
    except requests.RequestException as exc:
        connectivity[provider] = {
            "ok": False,
            "error_type": type(exc).__name__,
            "error": str(exc),
        }

connectivity

{'pubmed': {'status_code': 200,
  'ok': True,
  'final_url': 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=insomnia+slow+wave+EEG&retmode=json&retmax=1&tool=SleepAgent',
  'response_preview': '{"header":{"type":"esearch","version":"0.3"},"esearchresult":{"count":"225","retmax":"1","retstart":"0","idlist":["42331'},
 'europe_pmc': {'status_code': 200,
  'ok': True,
  'final_url': 'https://www.ebi.ac.uk/europepmc/webservices/rest/search?query=insomnia+slow+wave+EEG&format=json&pageSize=1',
  'response_preview': '{"version":"6.9","hitCount":4519,"nextCursorMark":"AoIIQPJj8yg1NTgyMDIxOA==","nextPageUrl":"https://www.ebi.ac.uk/europe'}}

## 调试一个指定 provider

先修改 `PROVIDER`，再逐格执行 client 构建、请求、结果和日志检查。需要进入源码时，可在对应源码行设置 IDE 断点，或在异常后执行 `%debug`。

In [13]:
# 7. 选择 provider
PROVIDER = "openalex"
assert PROVIDER in PROVIDERS
config["api"]["providers"][PROVIDER]

{'enabled': True,
 'base_url': 'https://api.openalex.org',
 'api_key_env': 'OPENALEX_API_KEY',
 'requests_per_second': 5}

In [14]:
# 8. 构建 client，检查 BaseAPIClient 的实际参数
client = build_client(PROVIDER, config, rate_limit_enabled=True)
{
    "client_type": type(client).__name__,
    "base_url": client.base.base_url,
    "timeout_seconds": client.base.timeout_seconds,
    "max_retries": client.base.max_retries,
    "fail_open": client.base.fail_open,
    "cache_enabled": client.base.cache.enabled if client.base.cache else None,
}

{'client_type': 'OpenAlexClient',
 'base_url': 'https://api.openalex.org',
 'timeout_seconds': 20.0,
 'max_retries': 1,
 'fail_open': False,
 'cache_enabled': False}

In [15]:
# 9. 发起单次真实请求；此处报错时，在下一个单元格执行 %debug
result = client.search(QUERY, max_results=MAX_RESULTS)
result

APISearchResult(provider='openalex', query='insomnia slow wave EEG', count=3, records=[APILiteratureRecord(provider='openalex', provider_id='https://openalex.org/W2028428817', paper_id='api_paper_8c32145cb1c8', title='Spectral characteristics of sleep EEG in chronic insomnia', abstract='To determine whether the spectral characteristics of the sleep electroencephalogram (EEG) of insomniacs differ from that of healthy subjects, we compared in each of the first four non-rapid eye movement (NREM) and rapid eye movement (REM) episodes: (a) the time courses of absolute power, averaged over the subjects in each group, for the delta, theta, alpha, sigma and beta frequency bands; (b) the relationship between these time courses; and (c) the overnight trend of integrated power in each frequency band. The results show that NREM power, for all frequencies below the beta range, has slower rise rates and reaches lower levels in the insomniac group, whereas beta power is significantly increased. In RE

In [16]:
# 10. 验证标准化结果
assert result.provider == PROVIDER
assert result.query == QUERY
assert result.count == len(result.records)
assert result.count <= MAX_RESULTS

[
    {
        "paper_id": record.paper_id,
        "title": record.title,
        "year": record.year,
        "doi": record.doi,
        "url": record.url,
    }
    for record in result.records
]

[{'paper_id': 'api_paper_8c32145cb1c8',
  'title': 'Spectral characteristics of sleep EEG in chronic insomnia',
  'year': 1998,
  'doi': 'https://doi.org/10.1046/j.1460-9568.1998.00189.x',
  'url': 'https://openalex.org/W2028428817'},
 {'paper_id': 'api_paper_1dc164ecd233',
  'title': 'Sleep, insomnia, and depression',
  'year': 2019,
  'doi': 'https://doi.org/10.1038/s41386-019-0411-y',
  'url': 'https://openalex.org/W2944504803'},
 {'paper_id': 'api_paper_b9ec191738f6',
  'title': 'Contribution of the circadian pacemaker and the sleep homeostat to sleep propensity, sleep structure, electroencephalographic slow waves, and sleep spindle activity in humans',
  'year': 1995,
  'doi': 'https://doi.org/10.1523/jneurosci.15-05-03526.1995',
  'url': 'https://openalex.org/W1537142000'}]

In [17]:
# 11. 检查底层请求日志（PubMed 正常会产生 search + summary 两条）
[log.model_dump(mode="json") for log in client.base.logs]

[{'provider': 'openalex',
  'endpoint': 'works',
  'query': 'insomnia slow wave EEG',
  'status_code': 200,
  'success': True,
  'elapsed_seconds': 2.7528,
  'cached': False,
  'error': None,
  'timestamp': '2026-06-30T09:25:58.723032+00:00'}]

如果第 8 格抛出异常，在下面新建一个临时 code cell 并执行：

```python
%debug
```

常用命令：`u`/`d` 切换调用栈，`p variable` 查看变量，`l` 查看附近源码，`q` 退出。

## 四个 provider 分别测试

以下单元格彼此独立。只执行需要检查的 provider；每格都会产生真实网络请求。

In [ ]:
# 12. OpenAlex
openalex_client = build_client("openalex", config, rate_limit_enabled=True)
openalex_result = openalex_client.search(QUERY, max_results=MAX_RESULTS)
openalex_result

In [ ]:
# 13. PubMed（建议设置 NCBI_EMAIL；API key 可选）
pubmed_client = build_client("pubmed", config, rate_limit_enabled=True)
pubmed_result = pubmed_client.search(QUERY, max_results=MAX_RESULTS)
pubmed_result

In [ ]:
# 14. Europe PMC
europe_pmc_client = build_client("europe_pmc", config, rate_limit_enabled=True)
europe_pmc_result = europe_pmc_client.search(QUERY, max_results=MAX_RESULTS)
europe_pmc_result

In [ ]:
# 15. Semantic Scholar（无 API key 时可能被限流）
semantic_scholar_client = build_client("semantic_scholar", config, rate_limit_enabled=True)
semantic_scholar_result = semantic_scholar_client.search(QUERY, max_results=MAX_RESULTS)
semantic_scholar_result

In [18]:
# 16. 汇总已经执行过的 provider；未执行的会显示 not run
summary = {}
for provider in PROVIDERS:
    variable_name = f"{provider}_result"
    provider_result = globals().get(variable_name)
    summary[provider] = (
        {"count": provider_result.count, "warnings": provider_result.warnings}
        if provider_result is not None
        else "not run"
    )
summary

{'openalex': 'not run',
 'pubmed': 'not run',
 'europe_pmc': 'not run',
 'semantic_scholar': 'not run'}